# 歌词分词，词性标注

In [1]:
import json
import pandas as pd

# import jieba
# import jieba.posseg as pseg
import thulac
from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [2]:
import sys
sys.path.append('..')

# 分词，词频与词性分析

In [4]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [5]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [3]:
thu = thulac.thulac(seg_only=False, filt=True) 

def process_lyrics_with_thulac(text, word_to_fix=None):
    if not text:
        return []
    
    # 2. 执行分词与词性标注
    # 返回格式为 [[word, pos], [word, pos], ...]
    words_with_pos = thu.cut(text)
    
    # 3. 过滤无意义字符与词性修正
    # thulac 的标点词性通常是 'w'
    filtered_data = []
    for word, pos in words_with_pos:
        word = word.strip()
        # 排除标点符号、空白字符
        if pos != 'w' and len(word) > 0:
            # 逻辑修正：word_to_fix 通常是修正词性
            if word_to_fix and word in word_to_fix:
                filtered_data.append((word, word_to_fix[word]))
            else:
                filtered_data.append((word, pos))
    
    # 4. 统计词频
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 5. 汇总信息
    # 建立 word -> pos 映射
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "word": word,
            "pos": word_pos_map[word],
            "freq": count
        })
    
    return sorted_results

Model loaded succeed


In [4]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [5]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word['song_id'] = df_word['song_id'].astype(str)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# main

In [6]:
# file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"
# file_path_prefix = "data/liuyuning/"
file_path_prefix = "data/newyear/"

In [7]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,213054628,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
1,106686506,000I3Ih84QiS9s,恭喜发财,NaN,刘德华,163,003aQYLo2x8izP,春节音乐,199630,003dNHXg186ZPk,202,1165939200,恭喜发财,2006-12-13
2,4826060,002NCGTk1dhWpG,过年好,NaN,天孪兄弟,15319,000absdi49XwcT,天生一对,429916,000MjDNI1CW5iU,186,1369670400,过年好,2013-05-28
3,228213715,000kdH8V2ieyeS,张灯结彩,NaN,"王二妮,阿宝","41449,5001","003lTJ1u3lA6hv,003oUwJ54CMqTT",猪福-孔雀群星贺新年,6094085,0045t5Q32NT70Q,244,1548950400,张灯结彩,2019-02-01
4,121641128,002NmMFi0YBMuS,欢乐中国年,NaN,孙悦,4448,004Tu0h03OnCTF,NaN,0,NaN,260,0,欢乐中国年,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,108706119,004esi930dmRrc,好年头好兆头,NaN,卓依婷,6351,002UILPL4dGoEz,祝福1,881291,0000DmIY3nG4Qn,362,1069603200,好年头好兆头,2003-11-24
62,105461811,001RQ9vB3RxxFW,喜庆临门,NaN,孙家山,166790,002sKdn31xAF4E,喜庆临门,1267543,000x9lRc2dUvXK,183,1451404800,喜庆临门,2015-12-30
63,200551951,0005TCNp4EOcEW,红红火火又一年,NaN,望海高歌,67405,001r9gSh4HW230,红红火火又一年,1829157,001nqm7L1QgOQh,223,1485014400,红红火火又一年,2017-01-22
64,456693581,004SZkCy1bU208,一年又一年,NaN,土豆王国小乐队,1398384,000vFOyZ2pFpVl,一年又一年,44723902,003FOIB523CM1Y,208,1703001600,一年又一年,2023-12-20


In [9]:
# 五月天需要使用word_to_fix
if file_path_prefix == "data/mayday/":
    df_word = lyric_words_process(file_path_prefix, word_to_fix)
else:
    df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [10]:
df_word

,song_id,word,pos,freq
0,213054628,来,v,32
1,213054628,运,v,25
2,213054628,好运,n,10
3,213054628,好,a,9
4,213054628,迎,v,7
...,...,...,...,...
4135,5122663,酿,v,2
4136,5122663,亲朋好友,id,2
4137,5122663,举起,v,2
4138,5122663,聚,v,2


In [11]:
df_merged = words_data_merge(df_word, df_songs)
df_merged

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,213054628,来,v,32,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
1,213054628,运,v,25,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
2,213054628,好运,n,10,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
3,213054628,好,a,9,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
4,213054628,迎,v,7,000MDaNK0krdFb,好运来,NaN,祖海,7352,002m2h2n4Veqcj,好运来,3908735,001KHxdj0upU5l,213,1074268800,好运来,2004-01-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4135,5122663,酿,v,2,000He5CA2bFuqP,中国吉祥,NaN,玖月奇迹,15581,003g372k4Vhnwc,中国吉祥,450757,002c4cfk0mmoNz,228,1388073600,中国吉祥,2013-12-27
4136,5122663,亲朋好友,id,2,000He5CA2bFuqP,中国吉祥,NaN,玖月奇迹,15581,003g372k4Vhnwc,中国吉祥,450757,002c4cfk0mmoNz,228,1388073600,中国吉祥,2013-12-27
4137,5122663,举起,v,2,000He5CA2bFuqP,中国吉祥,NaN,玖月奇迹,15581,003g372k4Vhnwc,中国吉祥,450757,002c4cfk0mmoNz,228,1388073600,中国吉祥,2013-12-27
4138,5122663,聚,v,2,000He5CA2bFuqP,中国吉祥,NaN,玖月奇迹,15581,003g372k4Vhnwc,中国吉祥,450757,002c4cfk0mmoNz,228,1388073600,中国吉祥,2013-12-27


In [12]:
df_merged.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

# 测试